# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rbblankson34/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/rbblankson34/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '55663b'. Skipping!
Property 'summary' already exists in node '41d4b7'. Skipping!
Property 'summary' already exists in node 'd8f806'. Skipping!
Property 'summary' already exists in node 'ec6678'. Skipping!
Property 'summary' already exists in node '21998f'. Skipping!
Property 'summary' already exists in node '5dfe62'. Skipping!
Property 'summary' already exists in node 'a3376f'. Skipping!
Property 'summary' already exists in node 'b8e451'. Skipping!
Property 'summary' already exists in node '96ca36'. Skipping!
Property 'summary' already exists in node '68b49d'. Skipping!
Property 'summary' already exists in node '5da8c6'. Skipping!
Property 'summary' already exists in node '4ce87d'. Skipping!
Property 'summary' already exists in node '03d431'. Skipping!
Property 'summary' already exists in node 'd34f9b'. Skipping!
Property 'summary' already exists in node '18968e'. Skipping!
Property 'summary' already exists in node '753006'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '68b49d'. Skipping!
Property 'summary_embedding' already exists in node '5dfe62'. Skipping!
Property 'summary_embedding' already exists in node '55663b'. Skipping!
Property 'summary_embedding' already exists in node 'a3376f'. Skipping!
Property 'summary_embedding' already exists in node 'b8e451'. Skipping!
Property 'summary_embedding' already exists in node '41d4b7'. Skipping!
Property 'summary_embedding' already exists in node '96ca36'. Skipping!
Property 'summary_embedding' already exists in node 'ec6678'. Skipping!
Property 'summary_embedding' already exists in node '4ce87d'. Skipping!
Property 'summary_embedding' already exists in node '21998f'. Skipping!
Property 'summary_embedding' already exists in node 'd8f806'. Skipping!
Property 'summary_embedding' already exists in node '5da8c6'. Skipping!
Property 'summary_embedding' already exists in node '03d431'. Skipping!
Property 'summary_embedding' already exists in node '18968e'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 712)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 712)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

✅ Answer:

The three types of query synthesizers in RAGAS are:

### 1. SingleHopSpecificQuerySynthesizer

- __What it does:__ Creates straightforward questions that can be answered by finding one specific piece of information from the documents
- __Simple explanation:__ Like asking "What color is the car?" - you only need to look in one place to find the answer
- __Example:__ "What is the main benefit of AI mentioned in the document?" or "Who is the author of this research?"
- __Use case:__ Testing if the RAG system can retrieve and return basic factual information correctly

### 2. MultiHopAbstractQuerySynthesizer

- __What it does:__ Generates complex questions requiring the system to connect ideas from multiple sources and think conceptually
- __Simple explanation:__ Like asking "How do these different trends relate to each other?" - you need to gather info from several places and think about the big picture
- __Example:__ "How do the emerging AI trends discussed relate to the ethical challenges mentioned across different sections?"
- __Use case:__ Testing if the RAG system can perform complex reasoning and synthesize information from multiple sources

### 3. MultiHopSpecificQuerySynthesizer

- __What it does:__ Creates questions that need specific facts from multiple locations but don't require abstract thinking
- __Simple explanation:__ Like asking "List all the red cars mentioned in chapters 1, 3, and 5" - you need to look in multiple places but for concrete facts
- __Example:__ "What are the three AI tools mentioned in section A and what are their specific features described in section C?"
- __Use case:__ Testing if the RAG system can accurately gather and combine specific factual information from multiple document sections

The distribution (50%, 25%, 25%) ensures that the synthetic test dataset has a good mix of simple and complex questions, allowing for comprehensive evaluation of the RAG system's ability to handle different types of queries.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,how does Ling and Imas 2025 say ChatGPT use im...,[Introduction ChatGPT launched in November 202...,The context discusses the rapid growth and ado...,single_hop_specifc_query_synthesizer
1,What is the significance of June 2024 in the c...,[Table 1: ChatGPT daily message counts (millio...,The report provides data ending on the 26th of...,single_hop_specifc_query_synthesizer
2,How do workplace professionals in high-paying ...,[Variation by Occupation Figure 23 presents va...,Variation by occupation shows that users in hi...,single_hop_specifc_query_synthesizer
3,How does the term 'Writing' relate to the usag...,[Conclusion This paper studies the rapid growt...,Writing is by far the most common work use of ...,single_hop_specifc_query_synthesizer
4,How does the rapid growth of ChatGPT and its i...,[<1-hop>\n\nConclusion This paper studies the ...,"The rapid growth of ChatGPT, with over 700 mil...",multi_hop_abstract_query_synthesizer
5,How does the privacy-preserving methodology su...,[<1-hop>\n\nConclusion This paper studies the ...,The privacy-preserving methodology used in stu...,multi_hop_abstract_query_synthesizer
6,How do the changes in message volume from June...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that between June 2024 and June...,multi_hop_abstract_query_synthesizer
7,How does the growth of ChatGPT usage in the US...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,"The context indicates that as of July 2025, ap...",multi_hop_specific_query_synthesizer
8,"Based on Handa et al., 2025, how ChatGPT usage...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Handa et al., 2025 report that nearly 80% of C...",multi_hop_specific_query_synthesizer
9,Based on the data showing that 18 billion mess...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT users were sending more ...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'ea1d9d'. Skipping!
Property 'summary' already exists in node 'c1af6a'. Skipping!
Property 'summary' already exists in node '28a7f6'. Skipping!
Property 'summary' already exists in node '8063fb'. Skipping!
Property 'summary' already exists in node 'd8a569'. Skipping!
Property 'summary' already exists in node '7edd33'. Skipping!
Property 'summary' already exists in node 'c82631'. Skipping!
Property 'summary' already exists in node '32e263'. Skipping!
Property 'summary' already exists in node 'd0a546'. Skipping!
Property 'summary' already exists in node '848bce'. Skipping!
Property 'summary' already exists in node '213fcd'. Skipping!
Property 'summary' already exists in node '08296e'. Skipping!
Property 'summary' already exists in node '77bceb'. Skipping!
Property 'summary' already exists in node '19f7fc'. Skipping!
Property 'summary' already exists in node '482f81'. Skipping!
Property 'summary' already exists in node 'deb990'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'd8a569'. Skipping!
Property 'summary_embedding' already exists in node 'c82631'. Skipping!
Property 'summary_embedding' already exists in node '28a7f6'. Skipping!
Property 'summary_embedding' already exists in node '8063fb'. Skipping!
Property 'summary_embedding' already exists in node 'c1af6a'. Skipping!
Property 'summary_embedding' already exists in node 'ea1d9d'. Skipping!
Property 'summary_embedding' already exists in node '7edd33'. Skipping!
Property 'summary_embedding' already exists in node 'd0a546'. Skipping!
Property 'summary_embedding' already exists in node '213fcd'. Skipping!
Property 'summary_embedding' already exists in node '77bceb'. Skipping!
Property 'summary_embedding' already exists in node '32e263'. Skipping!
Property 'summary_embedding' already exists in node '08296e'. Skipping!
Property 'summary_embedding' already exists in node '848bce'. Skipping!
Property 'summary_embedding' already exists in node 'deb990'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does O*NET relate to AI tools like ChatGPT...,[Introduction ChatGPT launched in November 202...,The provided context does not mention O*NET or...,single_hop_specifc_query_synthesizer
1,In Jun 2024 how many messages were non work an...,[Month Non-Work (M) (%) Work (M) (%) Total Mes...,"In June 2024, there were 238 million non-work ...",single_hop_specifc_query_synthesizer
2,What is the US in the context of message volum...,[Total daily counts are exact measurements of ...,Total daily counts are exact measurements of m...,single_hop_specifc_query_synthesizer
3,What are nonprofessional occupations according...,[Variation by Occupation Figure 23 presents va...,Nonprofessional occupations include administra...,single_hop_specifc_query_synthesizer
4,Considering the rapid adoption of ChatGPT sinc...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context indicates that since ChatGPT's lau...,multi_hop_abstract_query_synthesizer
5,How does the classification of ChatGPT usage i...,[<1-hop>\n\nTotal daily counts are exact measu...,The classification of ChatGPT usage into Pract...,multi_hop_abstract_query_synthesizer
6,how changin usage patterns within user cohorts...,[<1-hop>\n\nTotal daily counts are exact measu...,The context explains that total daily message ...,multi_hop_abstract_query_synthesizer
7,How does the conversation classification taxon...,[<1-hop>\n\nTotal daily counts are exact measu...,The conversation classification taxonomy devel...,multi_hop_abstract_query_synthesizer
8,how july 2025 chatgpt message counts compare t...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In june 2025, the total messages sent via Chat...",multi_hop_specific_query_synthesizer
9,Considering the rapid growth of ChatGPT usage ...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"The context indicates that by July 2025, appro...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways including performing workplace tasks by augmenting or automating human labor, producing writing, software code, spreadsheets, and other digital products. AI is used both as a co-worker producing output and as a co-pilot giving advice and improving human problem-solving productivity. People seek information and advice, but generative AI distinguishes itself through creating flexible, diverse digital outputs beyond traditional web search functions. Additionally, AI usage includes tasks related to relationships, personal reflection, games, and role play, though these are less common. Overall, AI is applied at work and outside work, supporting tasks that involve asking questions, doing tasks, or expressing ideas.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `dopeness_evaluator`:

✅ Answer:

Based on the notebook content, here's what each evaluator does:

### `qa_evaluator`:

- __What it does:__ Evaluates the quality of question-answering performance
- __Simple explanation:__ Checks if the RAG system's answer is correct and appropriate for the given question
- __Example:__ If asked "What is AI?", it evaluates whether the response accurately defines AI
- __Use case:__ Basic correctness assessment - ensuring the system provides factually accurate answers

### `labeled_helpfulness_evaluator`:

- __What it does:__ Evaluates how helpful the response is to the user, comparing it against the correct reference answer
- __Simple explanation:__ Judges whether the answer actually helps the user solve their problem or get the information they need
- __Example:__ Even if an answer is technically correct, this evaluator checks if it's presented in a way that's useful to the person asking
- __Use case:__ User experience assessment - ensuring responses are not just correct but genuinely useful and well-structured

### `dopeness_evaluator`:

- __What it does:__ Evaluates whether the response is engaging, creative, and avoids being generic
- __Simple explanation:__ Checks if the answer is "cool," "lit," or interesting rather than boring and template-like
- __Example:__ Instead of "AI is artificial intelligence," a "dope" answer might be "AI is like giving computers a brain that can think, learn, and solve problems just like humans do!"
- __Use case:__ Engagement assessment - ensuring the RAG system produces responses that are engaging and memorable rather than dry, robotic answers


## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'loyal-play-10' at:
https://smith.langchain.com/o/cbfba347-c08c-4a28-9b93-3cfe2f3af30b/datasets/30e38d3e-eab1-457f-93ec-986dba4e932a/compare?selectedSessions=528587f4-76ed-4439-8be6-3276c2e7ef88




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Hw is the chatGPT usage by July 2025, and how ...","By July 2025, ChatGPT had more than 700 millio...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,0,4.249474,0b96096b-081b-48e7-a586-030e9232e65d,f108fec7-a9dd-485c-83d0-6865c7c02c12
1,waht is the diffrence in messege count betwen ...,The difference in message count between June 2...,None,"In June 2024, there were 238 million non-work ...",1,1,0,2.356780,9c9c13f6-0c4d-41b7-a1ef-a6b32714cc3f,e3bc15d3-3daa-4ac5-94b3-7202f6988861
2,Considering the rapid growth of ChatGPT usage ...,"Based on the provided context, the increasing ...",None,"The context indicates that by July 2025, appro...",1,1,0,6.193357,4de473bc-19d4-41b7-b2b3-8ca837fb1052,94304ae2-4be6-476f-b6f7-3dea905e16dc
3,how july 2025 chatgpt message counts compare t...,"Based on the context, June 2025 had a total of...",None,"In june 2025, the total messages sent via Chat...",0,0,0,7.323739,5cc045a0-838c-4da1-8d8d-dc11695105cc,8b239442-49be-46db-ab52-d206f5b71df3
4,How does the conversation classification taxon...,The conversation classification taxonomy helps...,None,The conversation classification taxonomy devel...,1,1,0,2.290941,94dafe9a-5176-4091-b97d-76620d58cb42,b1330085-d98e-44fb-b48a-f38faacfdbd9
5,how changin usage patterns within user cohorts...,Changing usage patterns within user cohorts af...,None,The context explains that total daily message ...,1,1,0,2.720172,e509320a-bdad-44a7-8cbc-2f544c439980,9fdc1a9f-da20-4f4b-af1d-fdaa9a87e95d
6,How does the classification of ChatGPT usage i...,The classification of ChatGPT usage into Pract...,None,The classification of ChatGPT usage into Pract...,1,1,0,3.754606,bdf408ab-0a01-4526-86d1-65d99b05724b,7adc05b6-a1a6-466c-b517-dc3168a68a5d
7,Considering the rapid adoption of ChatGPT sinc...,"Based on the provided context, the societal ef...",None,The context indicates that since ChatGPT's lau...,1,1,0,6.668779,dae2766e-a578-4670-a1fb-7527caa4d95d,a3ea238e-a460-4ff3-b84f-e994eef07c7b
8,What are nonprofessional occupations according...,"Nonprofessional occupations, according to the ...",None,Nonprofessional occupations include administra...,1,1,0,0.991385,aab9ba5c-caba-4fd7-9876-3d38a1d426fc,2d4aaca8-9c43-4a21-b132-2f15cbba6317
9,What is the US in the context of message volum...,"Based on the provided context, the ""US"" refers...",None,Total daily counts are exact measurements of m...,1,1,0,2.035400,7859e2d3-db8b-4b93-9842-baf8571630ab,7fb6a26d-678a-428b-818e-d7bcc1552501


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

✅ Answer:

Modifying chunk size affects RAG performance in several important ways:

### __Information Density vs. Precision Trade-off:__

- __Smaller chunks (500 characters):__ More precise, focused pieces of information but may lack context
- __Larger chunks (1000 characters):__ More comprehensive context but potentially less precise retrieval

### __Retrieval Quality Impact:__

- __What it does:__ Changes how much context the retriever can access and return for each query
- __Simple explanation:__ Like choosing between reading individual sentences vs. entire paragraphs - bigger chunks give more complete thoughts but might include irrelevant info
- __Example:__ A 500-char chunk might contain "AI improves efficiency" while a 1000-char chunk includes "AI improves efficiency by automating repetitive tasks, reducing human error, and enabling 24/7 operations"
- __Use case:__ Larger chunks help answer complex questions that need more context, while smaller chunks are better for specific factual queries

### __Performance Implications:__

1. __Answer Completeness:__ Larger chunks provide more complete context, leading to more comprehensive answers
2. __Relevance:__ Smaller chunks may be more precisely relevant but could miss important surrounding context
3. __Processing Efficiency:__ Larger chunks mean fewer total chunks to search through, but each chunk takes more processing power
4. __Overlap Considerations:__ With the 50-character overlap, larger chunks maintain better continuity between adjacent pieces

### __Why the Change Improves "Dopeness":__

The increase from 500 to 1000 characters allows the system to retrieve more complete thoughts and context, enabling it to generate more comprehensive and engaging ("dope") responses rather than fragmented, incomplete answers.



In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

✅ Answer:

Modifying the embedding model from `text-embedding-3-small` to `text-embedding-3-large` affects RAG performance in several key ways:

### __Model Capability Differences:__

- __What it does:__ Changes how text is converted into numerical representations (vectors) that capture semantic meaning
- __Simple explanation:__ Like upgrading from a basic translator to an expert linguist - the larger model understands nuances and relationships better
- __Example:__ Both models might understand "car" and "automobile" are related, but the large model better captures subtle differences between "sedan," "SUV," and "sports car"
- __Use case:__ Better semantic understanding leads to more accurate retrieval of relevant documents

### __Performance Improvements:__

1. __Higher Dimensional Representations:__

   - `text-embedding-3-large` creates richer, more detailed vector representations
   - Captures more nuanced semantic relationships and context
   - Better distinguishes between similar but distinct concepts

2. __Improved Similarity Matching:__

   - More accurate calculation of semantic similarity between queries and document chunks
   - Better at finding relevant content even when exact keywords don't match
   - Reduces false positives and improves precision of retrieved chunks

3. __Enhanced Context Understanding:__

   - Better comprehension of domain-specific terminology and concepts
   - Improved handling of synonyms, related terms, and conceptual relationships
   - More effective at understanding the intent behind complex queries

### __Impact on RAG Quality:__

- __Retrieval Accuracy:__ More relevant chunks are retrieved for each query
- __Answer Quality:__ Better context leads to more accurate and comprehensive responses
- __Semantic Search:__ Improved ability to find conceptually related information even without exact keyword matches
- __Reduced Noise:__ Fewer irrelevant chunks retrieved, leading to cleaner, more focused answers

### __Trade-offs:__

- __Cost:__ Larger models are more expensive to run
- __Speed:__ Slightly slower processing due to increased complexity
- __Storage:__ Larger vector dimensions require more storage space




In [38]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [39]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [40]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [41]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Yo, here’s the ultra-dope lowdown from the context on how folks are stacking paper with AI:\n\nPeople ain’t just having AI do menial tasks—they’re tapping ChatGPT as their *secret weapon advisor* and *research assistant*. It’s like having a supercharged brain backup that boosts decision-making quality, especially in those knowledge-heavy gigs. This means workers are not just outsourcing grunt work to AI but *leveling up* their output through smarter moves and sharper choices.\n\nAlso, Collis and Brynjolfsson’s 2025 data drops a bomb—US users value generative AI so much they’d need nearly $98 to chill without it for a month. Translation? That’s a $97 billion+ annual stash of surplus value generated by using AI, much of it fuelled by smarter productivity, decision support, and turbocharged knowledge work.\n\nIn essence, AI transforms humans into productivity ninjas, letting them crank out more value, faster, and more creatively, which translates to real money-making mojo.\n\nThat’s not 

Finally, we can evaluate the new chain on the same test set!

In [42]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'abandoned-horn-90' at:
https://smith.langchain.com/o/cbfba347-c08c-4a28-9b93-3cfe2f3af30b/datasets/30e38d3e-eab1-457f-93ec-986dba4e932a/compare?selectedSessions=2298992a-11e7-4c3b-969a-38382f0de177




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,"Hw is the chatGPT usage by July 2025, and how ...","Yo, by July 2025, ChatGPT was absolutely blowi...",None,"By July 2025, ChatGPT had been used weekly by ...",1,1,1,5.299241,0b96096b-081b-48e7-a586-030e9232e65d,f1dc749f-8014-4ead-b6e2-8fac7819ee4a
1,waht is the diffrence in messege count betwen ...,"Yo, let’s crank those digits like a boss! 🔥\n\...",None,"In June 2024, there were 238 million non-work ...",1,1,1,5.276474,9c9c13f6-0c4d-41b7-a1ef-a6b32714cc3f,a7fe018a-1f2b-4858-a4ec-deedfd023fb2
2,Considering the rapid growth of ChatGPT usage ...,"Alright, here’s the skinny with some serious f...",None,"The context indicates that by July 2025, appro...",1,1,1,8.173173,4de473bc-19d4-41b7-b2b3-8ca837fb1052,7ce6472a-1433-402f-9495-d37292dd93c6
3,how july 2025 chatgpt message counts compare t...,"Alright, let’s blast off with some digital roc...",None,"In june 2025, the total messages sent via Chat...",0,0,1,6.325682,5cc045a0-838c-4da1-8d8d-dc11695105cc,16b15c8b-32eb-4bf0-b5cd-94bcd6cc37af
4,How does the conversation classification taxon...,"Yo, buckle up for some next-level insights on ...",None,The conversation classification taxonomy devel...,1,1,1,6.254649,94dafe9a-5176-4091-b97d-76620d58cb42,9f938cc5-8445-4fae-876d-57bd436e4f4e
5,how changin usage patterns within user cohorts...,"Alright, here’s the scoop with mad clarity and...",None,The context explains that total daily message ...,1,1,1,5.961215,e509320a-bdad-44a7-8cbc-2f544c439980,57a1a908-b273-4411-8424-1ccbe261b4ef
6,How does the classification of ChatGPT usage i...,"Yo, here’s the lowdown that makes ChatGPT’s vi...",None,The classification of ChatGPT usage into Pract...,1,1,1,4.919273,bdf408ab-0a01-4526-86d1-65d99b05724b,32931c79-2753-4c7b-9156-ee9790ee0ca3
7,Considering the rapid adoption of ChatGPT sinc...,"Alright, strap in — here’s the AI game-changer...",None,The context indicates that since ChatGPT's lau...,1,1,1,9.300699,dae2766e-a578-4670-a1fb-7527caa4d95d,e4fcf188-5d1f-4dd3-82e8-e62d0cb3091d
8,What are nonprofessional occupations according...,"Yo, buckle up for this: nonprofessional occupa...",None,Nonprofessional occupations include administra...,1,1,1,2.250595,aab9ba5c-caba-4fd7-9876-3d38a1d426fc,83656c23-8484-4bdf-bb07-38f3bd42fe74
9,What is the US in the context of message volum...,"Yo, here’s the juice straight from the AI stre...",None,Total daily counts are exact measurements of m...,1,0,1,4.813032,7859e2d3-db8b-4b93-9842-baf8571630ab,36f81e3f-29c9-490a-bc40-c5cb5173e08a


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

✅ Answer:


**Before improvement**:

![rag chain](img/rag_chain_1.png)

**After improvement**:

![rag chain](img/dopeness.png)


**comparison**

![comparison](img/comparison.png)

### Activity #3: Analysis of Chain Performance Differences

### Key Metric Changes Observed:

**1. QA Evaluator Performance (Correctness):**
- **Before:** 0.8333 (83.33%)
- **After:** 0.8333 (83.33%)
- **Why:** No improvement occurred despite the technical enhancements. The original RAG system was already performing at a high level for correctness. The larger embedding model (`text-embedding-3-large`) and increased chunk size (1000 vs 500 characters) didn't provide additional accuracy benefits, indicating that the baseline system was already retrieving sufficiently relevant and accurate information to answer questions correctly.

**2. Labeled Helpfulness Evaluator:**
- **Before:** 0.75 (75%)
- **After:** 0.75 (75%)
- **Why:** Helpfulness remained constant because the original system was already providing adequately useful responses to users. The technical improvements (larger chunks and better embeddings) didn't translate to increased practical utility, suggesting that the information retrieval quality was already sufficient for helping users solve their problems effectively.

**3. Dopeness Evaluator:**
- **Before:** 0.00 (0%)
- **After:** 1.00 (100%)
- **Why:** This metric showed perfect improvement, going from completely generic responses to maximally engaging ones. This dramatic change is entirely attributable to the explicit prompt modification that instructs the model to "Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses." The technical improvements had no impact here - it was purely the prompt engineering that transformed boring, template-like responses into creative, engaging answers.

### Technical Reasons for the Results:

**Embedding Model Upgrade & Chunk Size Increase:**
- While these improvements theoretically provide better semantic understanding and more comprehensive context, they didn't translate to measurable improvements in this evaluation
- The baseline system was already performing well enough that these enhancements didn't push the metrics higher
- This suggests diminishing returns - the original configuration was already adequate for the task complexity

**Prompt Engineering Impact:**
- The only measurable improvement came from explicitly instructing the model to avoid generic responses
- This demonstrates that response style and engagement can be dramatically improved through targeted prompt modifications
- Shows that technical retrieval improvements don't always correlate with user experience improvements

The results highlight that while technical optimizations are important, prompt engineering can have the most immediate and dramatic impact on specific evaluation criteria, especially those related to response quality and user engagement.